# 03 — Validation mAP Failure Analysis

Goal: find what is limiting **mAP@0.5** on validation before spending Kaggle submissions.

What to look for:

- **False negatives**: visible vehicles that the model misses. Usually fixed by label cleanup, more epochs, or threshold tuning.
- **False positives**: boxes on non-vehicles or unlabeled vehicles. If many are real vehicles, labels may be missing.
- **Localization errors**: correct class but IoU below 0.5. Usually loose labels, tiny boxes, or augmentation/training instability.
- **Class confusion**: good overlap but wrong class, especially `car` vs `van` and `truck` vs `bus`.
- **Confidence calibration**: true positives with low confidence and false positives with high confidence determine the best `predict.conf`.

Run this notebook after at least one `make train`. Set `RUN_VAL_PREDICT = True` once to cache validation predictions.


In [ ]:
from pathlib import Path
import os
import random
import sys


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle

from ua_detrac_starter.config import config_path, dataset_split_path, load_config, load_yaml

cfg, _ = load_config("config/competition.yaml")
dataset_yaml = config_path(cfg, "dataset_yaml", "config/dataset.yaml")
dataset_cfg = load_yaml(dataset_yaml)

val_images = dataset_split_path(dataset_yaml, dataset_cfg, "val")
val_labels = val_images.parent / "labels"
runs_root = config_path(cfg, "runs_detect_root", "runs/detect")
run_name = cfg.get("training", {}).get("run_name", "yolov8n_baseline")
weights = runs_root / run_name / "weights" / "best.pt"
pred_cache = REPO_ROOT / "experiments" / "cache" / f"{run_name}_val_predictions.csv"

CLASS_NAMES = {int(k): v for k, v in dataset_cfg.get("names", {}).items()}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IOU_MATCH_THRESHOLD = 0.50
RUN_VAL_PREDICT = False  # Set True once to run model inference and refresh pred_cache.

print("val_images:", val_images, val_images.is_dir())
print("val_labels:", val_labels, val_labels.is_dir())
print("weights:", weights, weights.is_file())
print("prediction cache:", pred_cache)


## Load Ground Truth

Comments while reading the tables:

- If a class is rare, its AP will be noisy and a few bad labels can dominate.
- Very small boxes often cap mAP because the model must localize tightly at IoU 0.5.
- Images with many boxes are high-value review targets because one image can contribute many errors.


In [ ]:
def list_images(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_EXTS)


def parse_yolo_label(label_path: Path, image_path: Path) -> list[dict]:
    rows = []
    if not label_path.is_file():
        return rows
    image = Image.open(image_path)
    image_width, image_height = image.size
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        class_id = int(parts[0])
        x_center, y_center, width, height = [float(value) for value in parts[1:]]
        rows.append({
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "label_path": str(label_path),
            "line_number": line_number,
            "class_id": class_id,
            "class_name": CLASS_NAMES.get(class_id, str(class_id)),
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
            "x1": max(0.0, x_center - width / 2),
            "y1": max(0.0, y_center - height / 2),
            "x2": min(1.0, x_center + width / 2),
            "y2": min(1.0, y_center + height / 2),
            "area": width * height,
            "image_width": image_width,
            "image_height": image_height,
        })
    return rows


gt_rows = []
for image_path in list_images(val_images):
    gt_rows.extend(parse_yolo_label(val_labels / f"{image_path.stem}.txt", image_path))

gt = pd.DataFrame(gt_rows)
print(f"validation images: {len(list_images(val_images)):,}")
print(f"ground-truth boxes: {len(gt):,}")
display(pd.crosstab(gt["class_name"], columns="boxes", margins=True) if not gt.empty else gt)
gt.head()


## Cache Validation Predictions

Set `RUN_VAL_PREDICT = True` only when you want to run inference. The cache lets you rerun all analysis cells without touching the GPU.

Look out for:

- Prediction cache from the wrong `run_name` or stale weights.
- Very low `conf` is intentional here (`0.001`) because mAP/AP needs the confidence ranking, not just final thresholded boxes.


In [ ]:
def result_to_rows(result, image_path: Path) -> list[dict]:
    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return []
    xywhn = boxes.xywhn.cpu().numpy()
    classes = boxes.cls.cpu().numpy().astype(int)
    confs = boxes.conf.cpu().numpy()
    rows = []
    for i in range(len(boxes)):
        x_center, y_center, width, height = [float(value) for value in xywhn[i]]
        class_id = int(classes[i])
        rows.append({
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "class_id": class_id,
            "class_name": CLASS_NAMES.get(class_id, str(class_id)),
            "confidence": float(confs[i]),
            "x_center": min(1.0, max(0.0, x_center)),
            "y_center": min(1.0, max(0.0, y_center)),
            "width": min(1.0, max(0.0, width)),
            "height": min(1.0, max(0.0, height)),
            "x1": max(0.0, x_center - width / 2),
            "y1": max(0.0, y_center - height / 2),
            "x2": min(1.0, x_center + width / 2),
            "y2": min(1.0, y_center + height / 2),
            "area": float(width) * float(height),
        })
    return rows


if RUN_VAL_PREDICT:
    if not weights.is_file():
        raise FileNotFoundError(f"Missing weights: {weights}")
    from tlc_ultralytics import YOLO

    pred_cache.parent.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights))
    image_paths = list_images(val_images)
    pred_rows = []
    for image_path in image_paths:
        results = model.predict(
            source=str(image_path),
            imgsz=int(cfg.get("training", {}).get("image_size", 640)),
            conf=0.001,
            iou=0.70,
            max_det=300,
            verbose=False,
        )
        pred_rows.extend(result_to_rows(results[0], image_path))
    pd.DataFrame(pred_rows).to_csv(pred_cache, index=False)
    print(f"wrote {pred_cache}")
else:
    print("RUN_VAL_PREDICT is False. Set it True to create/refresh the prediction cache.")

if pred_cache.is_file():
    preds = pd.read_csv(pred_cache)
else:
    preds = pd.DataFrame(columns=["image_id", "class_id", "confidence", "x1", "y1", "x2", "y2", "area"])
    print(f"No prediction cache yet: {pred_cache}")

print(f"predicted boxes loaded: {len(preds):,}")
preds.head()


## Match Predictions To Ground Truth

This approximates the competition mAP@0.5 logic on validation. It greedily matches predictions to same-class ground truth by confidence.

Interpretation:

- `true_positive`: correct class and IoU ≥ 0.5.
- `localization_error`: same class overlap exists, but IoU < 0.5.
- `class_confusion`: overlaps a vehicle but predicts the wrong class.
- `background_or_missing_label`: no strong overlap. Review these manually; some may be missing labels.
- `false_negative`: ground-truth box not matched by any prediction.


In [ ]:
def box_iou(a: pd.Series, b: pd.Series) -> float:
    ix1 = max(float(a.x1), float(b.x1))
    iy1 = max(float(a.y1), float(b.y1))
    ix2 = min(float(a.x2), float(b.x2))
    iy2 = min(float(a.y2), float(b.y2))
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, float(a.x2) - float(a.x1)) * max(0.0, float(a.y2) - float(a.y1))
    area_b = max(0.0, float(b.x2) - float(b.x1)) * max(0.0, float(b.y2) - float(b.y1))
    denom = area_a + area_b - inter
    return inter / denom if denom > 0 else 0.0


def average_precision(tp_flags: np.ndarray, fp_flags: np.ndarray, n_gt: int) -> float:
    if n_gt == 0:
        return np.nan
    tp_cum = np.cumsum(tp_flags)
    fp_cum = np.cumsum(fp_flags)
    recall = tp_cum / n_gt
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
    mrec = np.concatenate([[0.0], recall, [1.0]])
    mpre = np.concatenate([[0.0], precision, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    changing = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[changing + 1] - mrec[changing]) * mpre[changing + 1]))


def match_predictions(gt: pd.DataFrame, preds: pd.DataFrame, iou_threshold: float = 0.5):
    pred_records = []
    fn_records = []
    ap_rows = []

    gt = gt.copy().reset_index(drop=True)
    preds = preds.copy().reset_index(drop=True)
    gt["gt_id"] = np.arange(len(gt))
    preds["pred_id"] = np.arange(len(preds))

    for class_id in sorted(set(gt.get("class_id", [])) | set(preds.get("class_id", []))):
        class_gt = gt[gt["class_id"] == class_id]
        class_preds = preds[preds["class_id"] == class_id].sort_values("confidence", ascending=False)
        matched_gt_ids = set()
        tp_flags = []
        fp_flags = []

        for _, pred in class_preds.iterrows():
            same_image_gt = class_gt[class_gt["image_id"] == pred.image_id]
            best_iou = 0.0
            best_gt_id = None
            for _, gt_row in same_image_gt.iterrows():
                if int(gt_row.gt_id) in matched_gt_ids:
                    continue
                iou = box_iou(pred, gt_row)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_id = int(gt_row.gt_id)

            pred_record = pred.to_dict()
            pred_record["best_same_class_iou"] = best_iou
            pred_record["matched_gt_id"] = best_gt_id
            if best_iou >= iou_threshold and best_gt_id is not None:
                matched_gt_ids.add(best_gt_id)
                pred_record["match_type"] = "true_positive"
                tp_flags.append(1)
                fp_flags.append(0)
            else:
                pred_record["match_type"] = "false_positive"
                tp_flags.append(0)
                fp_flags.append(1)
            pred_records.append(pred_record)

        unmatched = class_gt[~class_gt["gt_id"].isin(matched_gt_ids)]
        for _, row in unmatched.iterrows():
            fn_record = row.to_dict()
            fn_record["match_type"] = "false_negative"
            fn_records.append(fn_record)

        ap_rows.append({
            "class_id": class_id,
            "class_name": CLASS_NAMES.get(int(class_id), str(class_id)),
            "gt_boxes": len(class_gt),
            "pred_boxes": len(class_preds),
            "tp": int(np.sum(tp_flags)),
            "fp": int(np.sum(fp_flags)),
            "fn": len(unmatched),
            "ap50": average_precision(np.array(tp_flags), np.array(fp_flags), len(class_gt)),
        })

    matched_preds = pd.DataFrame(pred_records)
    false_negatives = pd.DataFrame(fn_records)

    if not matched_preds.empty:
        refined_types = []
        for _, pred in matched_preds.iterrows():
            if pred.match_type == "true_positive":
                refined_types.append("true_positive")
                continue
            same_image_gt = gt[gt["image_id"] == pred.image_id]
            any_iou = 0.0
            any_class = None
            for _, gt_row in same_image_gt.iterrows():
                iou = box_iou(pred, gt_row)
                if iou > any_iou:
                    any_iou = iou
                    any_class = int(gt_row.class_id)
            if pred.best_same_class_iou >= 0.10:
                refined_types.append("localization_error")
            elif any_iou >= 0.50 and any_class != int(pred.class_id):
                refined_types.append("class_confusion")
            else:
                refined_types.append("background_or_missing_label")
        matched_preds["failure_type"] = refined_types

    return matched_preds, false_negatives, pd.DataFrame(ap_rows)


matched_preds, false_negatives, ap_table = match_predictions(gt, preds, IOU_MATCH_THRESHOLD)
map50 = ap_table["ap50"].dropna().mean() if not ap_table.empty else np.nan
print(f"approx validation mAP@0.5: {map50:.4f}" if not math.isnan(map50) else "mAP unavailable")
display(ap_table.round(4))


In [ ]:
if matched_preds.empty:
    print("No prediction matches available yet.")
else:
    display(matched_preds["failure_type"].value_counts().to_frame("count"))
    display(pd.crosstab(matched_preds["class_name"], matched_preds["failure_type"], margins=True))

if not false_negatives.empty:
    display(false_negatives["class_name"].value_counts().to_frame("false_negatives"))


## Review Visual Examples

Change `REVIEW_TYPE` to inspect the failure bucket you care about. Manual review is where mAP gains usually come from.

What to look for in each bucket:

- `false_negative`: Is the object tiny, occluded, mislabeled, or absent from train-like scenes?
- `localization_error`: Are labels too loose/tight, or does the model consistently miss full vehicle extent?
- `class_confusion`: Is the label defensible, or should the class guideline be clarified?
- `background_or_missing_label`: If the box is a real vehicle, the ground truth may be missing a label.


In [ ]:
def draw_boxes(ax, image_path: Path, gt_subset: pd.DataFrame, pred_subset: pd.DataFrame):
    image = Image.open(image_path).convert("RGB")
    image_width, image_height = image.size
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

    for _, row in gt_subset.iterrows():
        x = row.x1 * image_width
        y = row.y1 * image_height
        w = (row.x2 - row.x1) * image_width
        h = (row.y2 - row.y1) * image_height
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x, max(0, y - 4), f"GT {row.class_name}", color="black", backgroundcolor="lime", fontsize=8)

    for _, row in pred_subset.iterrows():
        x = row.x1 * image_width
        y = row.y1 * image_height
        w = (row.x2 - row.x1) * image_width
        h = (row.y2 - row.y1) * image_height
        label = f"P {row.class_name} {row.confidence:.2f}"
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor="yellow", linewidth=2))
        ax.text(x, y + h + 10, label, color="black", backgroundcolor="yellow", fontsize=8)


REVIEW_TYPE = "background_or_missing_label"  # true_positive, localization_error, class_confusion, background_or_missing_label, false_negative
MAX_IMAGES = 6

if REVIEW_TYPE == "false_negative":
    source = false_negatives.copy()
    image_ids = source["image_id"].drop_duplicates().head(MAX_IMAGES).tolist() if not source.empty else []
else:
    source = matched_preds[matched_preds.get("failure_type", pd.Series(dtype=str)) == REVIEW_TYPE].copy()
    source = source.sort_values("confidence", ascending=False) if "confidence" in source else source
    image_ids = source["image_id"].drop_duplicates().head(MAX_IMAGES).tolist() if not source.empty else []

if not image_ids:
    print(f"No examples for REVIEW_TYPE={REVIEW_TYPE}")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, image_id in zip(axes.flat, image_ids):
        image_path = Path(gt[gt["image_id"] == image_id]["image_path"].iloc[0]) if image_id in set(gt["image_id"]) else Path(preds[preds["image_id"] == image_id]["image_path"].iloc[0])
        gt_subset = gt[gt["image_id"] == image_id]
        pred_subset = matched_preds[matched_preds["image_id"] == image_id] if not matched_preds.empty else pd.DataFrame()
        draw_boxes(ax, image_path, gt_subset, pred_subset)
    for ax in axes.flat[len(image_ids):]:
        ax.axis("off")
    plt.tight_layout()


## Convert Findings To Actions

Use the dominant failure type to choose the next experiment:

| Dominant issue | First action | Second action |
|---|---|---|
| False negatives | inspect missed objects in 3LC; train longer | lower `predict.conf` only after checking FP cost |
| Background/missing-label candidates | inspect high-confidence examples; fix missing labels | retrain on revised tables |
| Localization errors | fix loose/tight labels; inspect tiny boxes | try more epochs / lower `lr0` |
| Class confusion | clarify class guidelines; fix wrong labels | inspect per-class AP before hyperparameters |
| Low AP only on one class | targeted data review for that class | class-specific threshold is not allowed in CSV, so fix data/model |
